In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
# Import Dash framework components for building the interactive dashboard UI
from dash import dcc, html
# Import Plotly for charting (pie chart)
import plotly.express as px
# Import DataTable for tabular display
from dash import dash_table
# Import callback dependencies for reactive UI updates
from dash.dependencies import Input, Output, State
import base64

# Configure Jupyter Dash
JupyterDash.infer_jupyter_proxy_config()

#import object from service_layer
from service_layer import AnimalService

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module2 import AnimalShelter

#import logging configuration
import logging_config


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aacuserPa55w0rD"
host = "127.0.0.1"
port = 27017
database = "aac"
collection = "animals"

# Connect to database via CRUD Module
db = AnimalShelter(username, password, host, port, database, collection)
# Inject the database object into the service layer (dependency injection)
service = AnimalService(db)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # Logo and unique identifier
    html.Div([
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
             style={'height': '100px'}),
    html.H4("Dashboard by Joseph Glista - CS340", style={'color': '#333'})
], style={'textAlign': 'center'}),


    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    # Interactive filter options
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True
        )
    ], style={'textAlign': 'center'}),
    
    html.Hr(),
    
    # Interactive data table
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         column_selectable=False,
                         row_selectable="single",
                         row_deletable=False,
                         selected_columns=[],
                         selected_rows=[],
                         page_action="native",
                         page_current=0,
                         page_size=10,
                         style_table={'overflowX': 'auto'},
                         style_cell={'textAlign': 'left', 'padding': '5px'}
                        ),
    html.Br(),
    html.Hr(),
# Set up dashboard so that chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################



# Added filter_query to the callback to validate filter string data    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value'),
              Input('datatable-id', 'filter_query')])
def update_dashboard(filter_type, filter_query):
    # Update the DataTable when the user selects a rescue filter or 
    # types into the DataTable's filter row.
    
    # Validate filter
    filter_type = service.validate_filter(filter_type)
    
    # Validate DataTable filter query
    filter_query = service.validate_table_filter(filter_query)
    
    # Get records from service layer
    records = service.get_animals(filter_type)
    
    # Convert records to DataFrame        
    df = pd.DataFrame.from_records(records)
    
    # Remove MongoDB internal ID field
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    
    # Apple DataTable filter_query if present
    if filter_query:
        try:
            df = df.to_dict(filter_query)
        except Exception as e:
            service.logger.warning(
                f"Filter query failed: '{filter_query}' - {e}"
            )
            # Fail gracefully by returning unfiltered data
            pass
    # Return updated table data
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.P("No data to display")]

    dff = pd.DataFrame(viewData)
    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Breed Distribution')
        )
    ]

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    # guard clause to prevent accessing a row that doesn't exist (no rows after filtering or empty list)
    if viewData is None or len(viewData) == 0:
        return [html.P("No data available for map")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    # make sure row is within bounds
    if row >= len(dff):
        return [html.P("Selected row is out of range")]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app on an alternative port to avoid conflicts with the original dashboard. (8051 vs original 8050)
app.run_server(mode='jupyterlab', port=8051)

2026-07-14 17:53:03,147 [INFO] AnimalService: Query built: {}
2026-07-14 17:53:31,045 [INFO] AnimalService: Query built: {'animal_type': 'Dog', 'breed': {'$in': ['German Shepherd', 'Border Collie']}}
2026-07-14 17:53:34,454 [INFO] AnimalService: Query built: {'animal_type': 'Dog', 'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever']}}
